# T3a on MAP01 — PPO, play-properly reward only (M6 acceptance, run 4)

**How to run (important):**
1. Settings: **Accelerator: GPU T4 x2**, **Internet: on**.
2. Click **Save Version** → choose **Save & Run All (Commit)** → **Save**. Do **not** run cells by hand.
3. You can close the browser. It takes **about 1.5–2 hours**.
4. When the version finishes, open it (Code → your notebook → Versions, or the link Kaggle shows) and scroll to the **last cell's output**. Copy it and paste it to Claude. Everything needed is printed there; no downloads.

What it runs: tests (PPO wiring, shaping, oracle fix) → two T3a seeds in parallel, one per T4, 2M steps each → an analysis of what the agent learned.

In [ ]:
import os
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
REPO_URL = "https://github.com/Satyajit-147/VizDoom.git"
BRANCH = "main"
WORKDIR = "/kaggle/working/rl-qa-tester"

In [ ]:
import subprocess
subprocess.run(["rm", "-rf", WORKDIR], check=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, WORKDIR], check=True)
os.chdir(WORKDIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -e ".[train,dev]"

In [ ]:
import sys, torch, sb3_contrib
print("python", sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.cuda.is_available(), torch.cuda.device_count(), "| sb3-contrib", sb3_contrib.__version__)

In [ ]:
!python -m pytest tests/test_ppo_smoke.py tests/test_shaping.py tests/test_env_units.py -q -k "not sweep"

In [ ]:
# Two T3a seeds in parallel, one per T4. Prints progress every 5 minutes into the log.
# Watchdog: a job that says RUN COMPLETE but does not exit within 3 minutes, or any job
# still running after 9 hours, gets its stacks dumped to its log and is killed.
import os, sys, time, signal, subprocess, pathlib
LOG = pathlib.Path("/kaggle/working/logs"); LOG.mkdir(parents=True, exist_ok=True)
base_env = {**os.environ, "SDL_VIDEODRIVER": "dummy", "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"}
procs = {}
for seed in (0, 1):
    out = open(LOG / f"t3a_seed{seed}.log", "w")
    procs[seed] = subprocess.Popen(
        [sys.executable, "scripts/train.py", "--config", "configs/t3a_map01.yaml", "--seed", str(seed),
         "--runs-dir", "/kaggle/working/runs", "--results-dir", "/kaggle/working/results"],
        stdout=out, stderr=subprocess.STDOUT, start_new_session=True,
        env={**base_env, "CUDA_VISIBLE_DEVICES": str(seed)})
start, done_at, last_print = time.time(), {}, 0.0
def lines(seed): return (LOG / f"t3a_seed{seed}.log").read_text(errors="replace").splitlines()
while any(p.poll() is None for p in procs.values()):
    time.sleep(30)
    now = time.time()
    for seed, p in procs.items():
        ls = lines(seed)
        if seed not in done_at and any(l.startswith("RUN COMPLETE") for l in ls):
            done_at[seed] = now
        stuck = seed in done_at and now - done_at[seed] > 180
        if p.poll() is None and (stuck or now - start > 9 * 3600):
            print(f"seed {seed}: {'finished but not exiting' if stuck else 'past 9 h'}: dumping stacks and killing", flush=True)
            os.kill(p.pid, signal.SIGUSR1); time.sleep(5)
            os.killpg(p.pid, signal.SIGKILL)
    if now - last_print >= 300:
        last_print = now
        for seed, p in procs.items():
            ls = lines(seed)
            prog = [l for l in ls if l.startswith("[")]
            state = "running" if p.poll() is None else f"exited ({p.returncode})"
            print(f"[{(now - start) / 60:5.0f} min] seed {seed} {state}: {(prog or ls or [''])[-1][:160]}", flush=True)
for seed in procs:
    print(f"\n--- seed {seed}: last log lines ---")
    print("\n".join(lines(seed)[-20:]))

In [ ]:
# ===== ANALYSIS: copy this cell's whole output to Claude =====
import json, pathlib, collections
W = pathlib.Path("/kaggle/working")
def rows(p): return [json.loads(l) for l in p.read_text().splitlines() if l] if p.exists() else []
for seed in (0, 1):
    run = W / f"runs/t3a_map01_seed{seed}"
    print(f"\n================ seed {seed} ================")
    for e in rows(run / "eval.jsonl"):
        m = e["maps"]["MAP01"]
        print(f"  eval {e['step']//1000:>5}k: goal {m['goal_rate']:.2f} ({m['episodes']} eps), return {m['mean_return']:6.2f}, length {m['mean_steps']:5.0f}")
    eps = rows(run / "episodes.jsonl")
    buckets = collections.defaultdict(list)
    for e in eps:
        buckets[e["step"] // 100_000].append(e)
    print(f"  training episodes: {len(eps)}")
    print("   steps   eps  key%  door%  goal%  death%  return  shaping  progress   len")
    for b, es in sorted(buckets.items()):
        n = len(es); part = lambda k: sum(e["parts"].get(k, 0) for e in es) / n
        key = sum(e["parts"].get("progress", 0) >= 1 for e in es) / n
        door = sum(e["parts"].get("progress", 0) >= 2 for e in es) / n
        goal = sum(e["outcome"] == "goal" for e in es) / n
        death = sum(e["outcome"] == "death" for e in es) / n
        ret = sum(e["return"] for e in es) / n
        ln = sum(e["steps"] for e in es) / n
        print(f"   {b*100:>4}k  {n:4d}  {key:4.0%}  {door:5.0%}  {goal:5.0%}  {death:6.0%}  {ret:6.2f}  {part('shaping'):7.2f}  {part('progress'):8.2f}  {ln:5.0f}")
    met = rows(run / "metrics.jsonl")
    if met:
        print("  speed:", met[-1]["steps_per_s"], "steps/s at the end; raw findings:", met[-1]["raw_findings"])
    rep = W / f"results/t3a_map01_seed{seed}/MAP01/report.json"
    if rep.exists():
        for f in json.loads(rep.read_text())["findings"]:
            print("  finding:", f["id"], f["kind"], "| room", f["room"], "| x", f["cluster_size"], "|", json.dumps(f["evidence"])[:250])
    final = rows(run / "eval.jsonl")[-1]["maps"]["MAP01"] if rows(run / "eval.jsonl") else None
    print("  M6 check (goal > 80%):", "PASS" if final and final["goal_rate"] > 0.8 else "FAIL")